# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# Section 1: Frame the lane as an ML task

ml_task_type = "Clustering (unsupervised learning)"

reason = """
My lane is a clustering task because the dataset does not contain a trusted label
that says which content archetype each page belongs to. The model should discover
groups of pages that have similar measured search-performance and engagement
characteristics.

The output will be a cluster ID for each page plus a profile of each cluster.
Those profiles can support decisions such as protecting, refreshing, improving,
monitoring, merging, or pruning content. The cluster ID itself is not a final
business action; a human reviewer should interpret each cluster before acting.
"""

print("ML task type:", ml_task_type)
print(reason)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# Section 2: Define the target or proxy

target = None

proxy_features = [
    "impressions_90d",
    "ctr",
]

target_explanation = """
This is unsupervised learning, so there is no observed outcome label to predict.
The model will create a cluster assignment from similarities in measured page-level
features. In this first framing, 90-day impressions and CTR are the confirmed
performance proxies from the Week 1 dataset.

The final feature list may expand after data inspection, but only features that
are available at decision time, interpretable, and safe to use will be included.
URLs, client names, raw private queries, and manually assigned action labels will
not be used as clustering features.

The learned cluster number is arbitrary. For example, Cluster 0 does not
automatically mean 'good' or 'bad'. Each cluster must be profiled and given a
descriptive name after training.
"""

print("Observed prediction target:", target)
print("Current proxy features:", proxy_features)
print(target_explanation)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# Section 3: Choose one success metric

success_metric = "Silhouette score"
minimum_acceptable_score = 0.40

metric_explanation = f"""
Primary metric: {success_metric}

The silhouette score measures whether pages are closer to other pages in their
own cluster than to pages in neighboring clusters. It ranges from -1 to 1.
Higher values indicate more compact and better-separated clusters.

For this exploratory project, a score of at least {minimum_acceptable_score:.2f}
will be treated as a useful initial result. This is a directional threshold,
not a universal guarantee. I will also check cluster sizes and profiles so that
a mathematically strong result is not accepted if it produces tiny, unstable,
or uninterpretable groups.
"""

print(metric_explanation)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# Section 4: Load the real dataframe and show the unit of analysis

from pathlib import Path
import pandas as pd

candidate_paths = [
    Path("content_refresh_anonymized.csv"),
    Path("data/content_refresh_anonymized.csv"),
    Path("../data/content_refresh_anonymized.csv"),
    Path("../../data/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is None:
    searched = "\n".join(f"- {p}" for p in candidate_paths)
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. "
        "Place it in one of these locations:\n" + searched
    )

df = pd.read_csv(data_path)

required_columns = ["impressions_90d", "ctr"]
missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    raise ValueError(
        "The dataset is missing required columns: " + ", ".join(missing_columns)
    )

# Each row represents one content page.
lane_df = df.copy()

print("Data source:", data_path)
print("Unit of analysis: one content page")
print("Number of rows/pages:", f"{len(lane_df):,}")
print("Number of columns:", lane_df.shape[1])
print("Duplicate full rows:", int(lane_df.duplicated().sum()))
print("Missing impressions_90d:", int(lane_df["impressions_90d"].isna().sum()))
print("Missing ctr:", int(lane_df["ctr"].isna().sum()))

display(lane_df.head())

# A small numerical view of the two confirmed clustering proxies.
display(
    lane_df[required_columns]
    .describe()
    .T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
)


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# Section 5: Explain why ML is more suitable than one fixed rule

fixed_rule_example = (
    "A fixed rule might label every page with low impressions as weak, "
    "but that could mix new pages, niche pages, high-CTR pages, and genuinely "
    "underperforming pages into the same group."
)

why_ml = """
Content performance is multi-dimensional. Impressions, CTR, engagement, age,
and other available measurements can interact in different ways. A page may
have high impressions but low CTR, low impressions but strong CTR, or moderate
values across several metrics. Several useful archetypes may therefore exist
instead of one clean boundary.

A long chain of if-statements would require hand-picked thresholds, would be
sensitive to scale and skew, and could miss combinations that were not defined
in advance. Clustering can search for recurring multivariate patterns across
the inventory.

ML does not automatically make the decision correct. The clusters will still
need stability checks, clear profiles, descriptive names, and human review.
The result is decision support based on observed measurements, not causal proof
and not a prediction of Google's algorithm.
"""

print("Example limitation of a fixed rule:")
print(fixed_rule_example)
print("\nWhy clustering is useful:")
print(why_ml)


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled with clear thinking and supporting code
- [ ] The notebook runs top to bottom with no errors after the CSV is placed in the repo
- [x] No client names, URLs, or private queries are included
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Commit under `work/notebooks/` and submit the repository URL on the card
